In [4]:
from datetime import date
from data_engineering.filters import pia_or_american_ladd_icao
from data_engineering.flights.flight_type import FlightIndex, flights_df_to_flights, get_flights
import polars as pl
start_dt = date(2025, 1, 1)
end_dt = date(2026, 7, 1)
df_flights = get_flights(start_dt, end_dt)
df_flights = df_flights.filter(pia_or_american_ladd_icao())
flights = flights_df_to_flights(df_flights)
flight_index = FlightIndex(flights, None)

In [12]:
import pickle
path = '/Volumes/T2-SSD/planequery/data/tests/test_pia/test_real_flights_3/test_pia_test_real_flights_3_20260826_173714/mappings.pkl'
with open(path, "rb") as f:
    mappings = pickle.load(f)

In [13]:
from pia.check_mappings import check_mappings
failed_icaos = check_mappings(flight_index, mappings)
failed_icaos

Flights a03bff_2025-09-24_13-25 and a03bc5_2025-09-24_11-34 overlap in time but are assigned to the same real aircraft a11784
Flights a03bff_2025-09-24_14-45 and a03bc5_2025-09-24_11-34 overlap in time but are assigned to the same real aircraft a11784
Flights a03bc5_2025-09-24_11-34 and a03bff_2025-09-24_13-25 overlap in time but are assigned to the same real aircraft a11784
Flights a03bc5_2025-09-24_11-34 and a03bff_2025-09-24_14-45 overlap in time but are assigned to the same real aircraft a11784
Flights a11784_2025-04-11_20-53 and a03bc5_2025-04-17_12-59 have a discontinuity in airports: KUZA -> KTDF 
Flights a03bc5_2025-05-19_17-04 and a03bff_2025-06-13_19-33 have a discontinuity in airports: KCHS -> KUZA 
Flights a7c742_2025-05-21_18-54 and a03c53_2025-05-24_19-41 have a discontinuity in airports: KRBW -> KCUB 
Flights a03c53_2025-06-01_19-44 and a7c742_2025-06-02_19-05 have a discontinuity in airports: KCUB -> KRBW 
Flights a4a3e8_2025-10-28_00-00 and a03d26_2025-10-28_17-32 have

{'a03bc5',
 'a03bff',
 'a03c53',
 'a03d26',
 'a03d63',
 'a0919f',
 'a091ab',
 'a11784',
 'a4a3e8',
 'a7c742'}

In [ ]:
from pia.check_mappings import real_icao_to_pia_icaos_from_flight_mapping, remove_failed_icaos
real_icao_to_pia_icaos = real_icao_to_pia_icaos_from_flight_mapping(mappings)

In [15]:
from pia.check_mappings import stats_on_mappings
stats = stats_on_mappings(flight_index, real_icao_to_pia_icaos)

In [ ]:
from pia.ground_truth import get_ground_truth_mappings
df_groundtruth = get_ground_truth_mappings()

pia_icao,registration,real_icao
str,str,str


In [17]:
stats_df = pl.DataFrame([
    {"icao": icao, **values}
    for icao, values in stats.items()
])
groundtruth_pia_icaos = set(df_groundtruth["pia_icao"].to_list())

stats_df = (
    stats_df.with_columns(
        pl.col("icao")
        .map_elements(
            lambda x: sorted(real_icao_to_pia_icaos.get(x, set())),
            return_dtype=pl.List(pl.Utf8),
        )
        .alias("pia_icaos"),
    )
    .with_columns(
        pl.col("pia_icaos")
        .list.eval(pl.element().filter(pl.element().is_in(groundtruth_pia_icaos)))
        .alias("groundtruth_icaos")
    )
)
with pl.Config(tbl_rows=-1):
    display(stats_df.sort("num_transitions", descending=True))

icao,num_transitions,num_flights,pia_icaos,groundtruth_icaos
str,i64,i64,list[str],list[str]
"""a11784""",14,425,"[""a03bc5"", ""a03bff""]","[""a03bff""]"
"""a4a3e8""",12,203,"[""a03d26"", ""a03d63"", … ""a091ab""]","[""a091ab""]"
"""ad5288""",5,137,"[""a03b79"", ""a03b9a"", … ""a03c72""]",[]
"""a835af""",4,110,"[""a039aa"", ""a03bed"", ""a03c08""]",[]
"""a29625""",3,1221,"[""a038c0""]","[""a038c0""]"
"""aa6a85""",3,77,"[""a0399e"", ""a03cbd"", ""a03d10""]",[]
"""ab30f5""",3,142,"[""a039ac"", ""a03c7c"", ""a091a6""]","[""a091a6""]"
"""ac6155""",3,10,"[""a03bf7"", ""a03cdb"", ""a10434""]",[]
"""a7c742""",3,105,"[""a03c53""]",[]


In [21]:
icao = "a29625"
icaos = [icao] + list(real_icao_to_pia_icaos[icao])

In [23]:
import polars as pl
icaos = ["a0f15e", "a0fdfa"]
COLUMNS = ["icao", "pia", "callsign", "takeoff_time", "landing_time", "takeoff_airport_ident", "landing_airport_ident"]
with pl.Config(tbl_rows=-1):
    display(df_flights.filter(pl.col("icao").is_in(icaos)).sort("takeoff_time").select(COLUMNS))

icao,pia,callsign,takeoff_time,landing_time,takeoff_airport_ident,landing_airport_ident
str,bool,str,datetime[ms],datetime[ms],str,str
"""a0fdfa""",true,"""FFL436""",2025-01-21 17:57:22.610,2025-01-21 18:03:46.190,"""KTEB""","""KLGA"""
"""a0fdfa""",true,"""FFL436""",2025-01-21 18:06:54.210,2025-01-21 18:14:27.650,"""KLGA""","""KTEB"""
"""a0fdfa""",true,"""FFL436""",2025-01-22 22:29:12.230,2025-01-22 22:35:02.740,"""KTEB""","""KLGA"""
"""a0fdfa""",true,"""FFL436""",2025-01-22 23:49:59.010,2025-01-22 23:56:06.030,"""KLGA""","""KTEB"""
"""a0fdfa""",true,"""FFL436""",2025-01-25 11:59:52.260,2025-01-25 13:52:56.050,"""KTEB""","""KPHF"""
"""a0fdfa""",true,"""FFL436""",2025-01-25 14:38:22.180,2025-01-25 15:32:50.780,"""KPHF""","""KGSB"""
"""a0fdfa""",true,"""FFL436""",2025-01-25 17:43:48.050,2025-01-25 19:02:27.950,"""SC65""","""KJAX"""
"""a0fdfa""",true,"""FFL436""",2025-01-25 20:09:40.320,2025-01-25 21:53:12.230,"""KJAX""","""KPBI"""
"""a0f15e""",false,"""N16WW""",2025-01-26 13:40:41.020,2025-01-26 19:52:31.200,"""KPBI""","""MBPV"""


pia_icao,registration,real_icao
str,str,str
